In [59]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=c25ba9ba9a662f165fc8fd2638977a3286541b43be6de3a19fcd6074b13dab7b
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [60]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
import re
import json
import torch
from datasets import load_dataset
from nltk.translate.bleu_score import sentence_bleu
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from transformers import DataCollatorWithPadding
from datasets import Dataset
from torch.utils.data import DataLoader
import pandas as pd

In [ ]:
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

In [55]:
ds = load_dataset("tatsu-lab/alpaca")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [56]:
ds['train'][0]

{'instruction': 'Give three tips for staying healthy.',
 'input': '',
 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}

In [57]:
len(ds['train'])

52002

In [61]:
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

In [62]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [63]:
def get_scores(target, result):
  # Bleu
  score_bleu = sentence_bleu([target.split()], result.split())

  # Semantic Similiarity
  emb1 = embedding_model.encode(target, convert_to_tensor=True)
  emb2 = embedding_model.encode(result, convert_to_tensor=True)

  score_similiarity = util.cos_sim(emb1, emb2).item()

  # Rouge L
  score_rouge_l = scorer.score(target, result)
  return score_bleu, score_similiarity, score_rouge_l['rougeL'].fmeasure

In [64]:
target = "The capital of Indonesia is Jakarta."
result = "Jakarta is the capital city of Indonesia."

get_scores(target, result)

/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

(1.4740564900137075e-231, 0.9616143703460693, 0.6153846153846153)

In [66]:
# Tokenization
all_features = []
subset = ds["train"].select(range(100))

for item in subset:
    messages = [{"role": "user", "content": item["instruction"]}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=2048,
        return_tensors=None
    )
    all_features.append(tokenized)

new_dataset = Dataset.from_list(all_features)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
dataloader = DataLoader(
    new_dataset,
    batch_size=4,
    collate_fn=collator
)

In [72]:
def get_last_assistant(text: str):
    pattern = r"<\|im_start\|>assistant\s*(.*?)<\|im_end\|>"
    matches = re.findall(pattern, text, flags=re.DOTALL)

    if matches:
        return matches[-1].strip()

    fallback_pattern = r"<\|im_start\|>assistant\s*(.*)$"
    fallback_matches = re.findall(fallback_pattern, text, flags=re.DOTALL)

    if fallback_matches:
        return fallback_matches[-1].strip()

    return text

In [75]:
special_tokens = list(tokenizer.special_tokens_map.values())

In [76]:
def clean(text):
    for tok in special_tokens:
        text = text.replace(tok, "")
    return text.strip()

In [84]:
# for i, batch in enumerate(dataloader):
#     print(i, batch)

In [85]:
results = []

for i, batch in enumerate(dataloader):
    print(i)
    batch = {k: v.to(model.device) for k, v in batch.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **batch,
            max_new_tokens=128
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)

    results_raw = [get_last_assistant(x) for x in decoded]
    results.extend([clean(r) for r in results_raw])

0


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


1


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


2


...

22


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


23


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


24


In [74]:
print(decoded[0])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Give three tips for staying healthy.<|im_end|>
<|im_start|>assistant
Certainly! Here are three effective tips for maintaining good health:

1. **Maintain a Balanced Diet**: Eating a variety of nutritious foods is crucial for overall health. Focus on incorporating plenty of fruits, vegetables, whole grains, and lean proteins into your diet. Limit the intake of processed foods, sugars, and unhealthy fats. Staying hydrated is also important, so make sure to drink enough water throughout the day.

2. **Regular Exercise**: Aim for at least 150 minutes of moderate aerobic activity or 75 minutes of vigorous activity each week, along with muscle-strengthening exercises on two or more days a week


In [86]:
columns = []

for i in range(len(results)):
  result = results[i]
  target = subset[i]['output']
  score_bleu, score_similiarity, score_rouge_l = get_scores(target, result)
  columns.append([result, target, score_bleu, score_similiarity, score_rouge_l])

df = pd.DataFrame(
    columns,
    columns=["result", "target", "bleu", "similarity", "rouge_l"]
)

/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

In [87]:
df

,result,target,bleu,similarity,rouge_l
0,Certainly! Here are three tips for maintaining...,1.Eat a balanced diet and make sure to include...,2.953846e-02,0.525081,0.194030
1,"The three primary colors are Red, Green, and B...","The three primary colors are red, blue, and ye...",3.553158e-02,0.742894,0.163636
2,An atom is the fundamental unit of matter and ...,"An atom is made up of a nucleus, which contain...",2.200805e-78,0.820048,0.319018
3,Reducing air pollution is a complex but crucia...,There are a number of ways to reduce air pollu...,1.721931e-78,0.860975,0.267516
4,"<tool_call>user\nSure, that would be interesti...",I had to make a difficult decision when I was ...,2.350744e-158,0.135479,0.077922
...,...,...,...,...,...
95,The phrase you want translated is not provided...,Je te manque.,0.000000e+00,0.312821,0.000000
96,"<|im_start|>user\nSure, could you explain what...",API stands for Application Programming Interfa...,1.316670e-233,0.544035,0.049383
97,"To compute the area of a rectangle, you use th...",The area of the rectangle is 50 cm2.,5.633617e-02,0.744001,0.197183
98,"As an AI, I always strive for up-to-date and a...",The capital of Spain is Madrid.,1.472821e-01,0.708667,0.375000
